In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [4]:
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=3020,
    stratify=y
)

print(f"Trained data: {X_train.shape[0]} samples | Test data: {X_test.shape[0]} samples")

Trained data: 105 samples | Test data: 45 samples


## Data Preparation

In [9]:
train_counts = np.bincount(y_train)
test_counts = np.bincount(y_test)

for i, name in enumerate(target_names):
    print(f"Class {i} ({name}): Train = {train_counts[i]}, Test = {test_counts[i]}")

Class 0 (setosa): Train = 35, Test = 15
Class 1 (versicolor): Train = 35, Test = 15
Class 2 (virginica): Train = 35, Test = 15


In [10]:
mean_train = np.mean(X_train, axis=0)
std_train = np.std(X_train, axis=0, ddof=0)

X_train_std = (X_train - mean_train) / std_train
X_test_std = (X_test - mean_train) / std_train

n_train = X_train_std.shape[0]
n_test = X_test_std.shape[0]

A_train = np.hstack([np.ones((n_train, 1)), X_train_std])
A_test = np.hstack([np.ones((n_test, 1)), X_test_std])

Y_train = np.zeros((n_train, 3))
Y_train[np.arange(n_train), y_train] = 1.0

print(f"A_train: {A_train.shape}, A_test: {A_test.shape}, Y_train: {Y_train.shape}")

A_train: (105, 5), A_test: (45, 5), Y_train: (105, 3)


## Least-Squares Classifier

In [11]:
ATA = A_train.T @ A_train
ATY = A_train.T @ Y_train
W_ls = np.linalg.solve(ATA, ATY)

def predict_ls(A, W):
    scores = A @ W
    return np.argmax(scores, axis=1)

y_pred_ls_train = predict_ls(A_train, W_ls)
y_pred_ls_test = predict_ls(A_test, W_ls)

acc_ls_train = np.mean(y_pred_ls_train == y_train)
acc_ls_test = np.mean(y_pred_ls_test == y_test)

print(f"Training Accuracy : {acc_ls_train * 100:.2f}%")
print(f"Test Accuracy     : {acc_ls_test * 100:.2f}%")

Training Accuracy : 86.67%
Test Accuracy     : 82.22%


## Softmax Regression

In [12]:
def compute_softmax_loss(A, Y, W, lam=1.0):
    n = A.shape[0]
    Z = A @ W
    z_max = np.max(Z, axis=1, keepdims=True)
    log_sum_exp = z_max + np.log(np.sum(np.exp(Z - z_max), axis=1, keepdims=True))
    log_P = Z - log_sum_exp

    cross_entropy = -np.sum(Y * log_P) / n
    l2_reg = (lam / (2.0 * n)) * np.sum(W[1:, :] ** 2)
    return cross_entropy + l2_reg

W_sm = np.zeros((5, 3))
initial_loss = compute_softmax_loss(A_train, Y_train, W_sm, lam=1.0)

lr = 0.1
num_iterations = 10000
lam = 1.0

for _ in range(num_iterations):
    Z = A_train @ W_sm
    z_max = np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z - z_max)
    P = exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    grad = (A_train.T @ (P - Y_train)) / n_train

    grad_reg = np.zeros_like(W_sm)
    grad_reg[1:, :] = (lam / n_train) * W_sm[1:, :]

    W_sm -= lr * (grad + grad_reg)

final_loss = compute_softmax_loss(A_train, Y_train, W_sm, lam=1.0)

def predict_softmax(A, W):
    Z = A @ W
    return np.argmax(Z, axis=1)

y_pred_sm_train = predict_softmax(A_train, W_sm)
y_pred_sm_test = predict_softmax(A_test, W_sm)

acc_sm_train = np.mean(y_pred_sm_train == y_train)
acc_sm_test = np.mean(y_pred_sm_test == y_test)

print(f"Initial Loss (W = 0) : {initial_loss:.6f}")
print(f"Final Loss           : {final_loss:.6f}")
print(f"Training Accuracy    : {acc_sm_train * 100:.2f}%")
print(f"Test Accuracy        : {acc_sm_test * 100:.2f}%")

Initial Loss (W = 0) : 1.098612
Final Loss           : 0.240405
Training Accuracy    : 96.19%
Test Accuracy        : 95.56%


## Results and Reporting

In [13]:
print(f"Softmax Initial Loss : {initial_loss:.6f}")
print(f"Softmax Final Loss   : {final_loss:.6f}")
print(f"Least-Squares -> Train Acc: {acc_ls_train * 100:.2f}%, Test Acc: {acc_ls_test * 100:.2f}%")
print(f"Softmax       -> Train Acc: {acc_sm_train * 100:.2f}%, Test Acc: {acc_sm_test * 100:.2f}%")

Softmax Initial Loss : 1.098612
Softmax Final Loss   : 0.240405
Least-Squares -> Train Acc: 86.67%, Test Acc: 82.22%
Softmax       -> Train Acc: 96.19%, Test Acc: 95.56%


In [14]:
def print_misclassified(model_name, y_true, y_pred):
    mis_idx = np.where(y_true != y_pred)[0]
    print(f"{model_name} Misclassified Test Samples (Total: {len(mis_idx)}):")
    if len(mis_idx) == 0:
        print("None")
    else:
        for idx in mis_idx:
            print(f"- Index {idx:2d}: True = {target_names[y_true[idx]]}, Predicted = {target_names[y_pred[idx]]}")

print_misclassified("Least-Squares Classifier", y_test, y_pred_ls_test)
print_misclassified("Softmax Regression", y_test, y_pred_sm_test)

Least-Squares Classifier Misclassified Test Samples (Total: 8):
- Index  5: True = virginica, Predicted = versicolor
- Index  6: True = virginica, Predicted = versicolor
- Index 18: True = versicolor, Predicted = virginica
- Index 28: True = virginica, Predicted = versicolor
- Index 29: True = versicolor, Predicted = virginica
- Index 32: True = versicolor, Predicted = virginica
- Index 35: True = versicolor, Predicted = virginica
- Index 41: True = versicolor, Predicted = virginica
Softmax Regression Misclassified Test Samples (Total: 2):
- Index  5: True = virginica, Predicted = versicolor
- Index  6: True = virginica, Predicted = versicolor
